# Task 2: Predicting severity metrics
Run this notebook to predict the severity (UPDRSM). 

The structure of this notebook is as follows: 
1. Data setup: loading features, metadata, splits
2. Training helper: function to train models and save outputs
3. Mean baseline 
4. Multiple linear regression
5. Random forest
6. 1D-CNN

## 1. Data setup

In [ ]:
from pathlib import Path
import sys
import json
import numpy as np
import pandas as pd
import joblib
from sklearn.dummy import DummyRegressor

ROOT = Path.cwd().resolve()
if not (ROOT / "src").is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

# CHECK PATH WHEN COMPLETE
FEATURES_PATH = ROOT / "data" / "features.csv"
SPLITS_PATH = ROOT / "data" / "splits.json"

# output directory
OUT = ROOT / "presentation_material" / "task2"
(OUT / "predictions").mkdir(parents=True, exist_ok=True)
(OUT / "weights").mkdir(parents=True, exist_ok=True)

# import functions from models 
from src.data import load_metadata
from src.models.linreg import make_linear_regression
from src.models.rf import make_random_forest

In [ ]:
# load data 
metadata = load_metadata().rename(columns={"ID": "subject"})
features = pd.read_csv(FEATURES_PATH).rename(
    columns={"ID": "subject", "Unnamed: 0": "subject"}
)

# merge metadata and features
patients = features.merge(
    metadata[["subject", "Group", "UPDRSM"]],
    on="subject", how="inner", validate="many_to_one"
)
patients = patients.loc[
    patients["Group"].eq("Pt") & patients["UPDRSM"].notna()
].reset_index(drop=True)

feature_columns = [
    "stride_time", "stride_cv", "swing_cv",
    "swing_asymmetry", "peak_force"
]

# create X and Y 
X = patients[feature_columns]
y = patients["UPDRSM"].astype(float)
print("Patients:", patients["subject"].nunique(), "| Feature matrix:", X.shape)
display(X.head())

In [ ]:
# unsure of what this does. need to check
with open(SPLITS_PATH) as f:
    assignment = json.load(f)["task2"]

test_subjects = set(assignment["test"])
folds = []
for fold in assignment["folds"]:
    train_ids, val_ids = set(fold["train"]), set(fold["val"])
    # Essential safety check: keep the held-out test subjects separate.
    assert train_ids.isdisjoint(val_ids | test_subjects)
    assert val_ids.isdisjoint(test_subjects)
    train_idx = np.flatnonzero(patients["subject"].isin(train_ids).to_numpy())
    val_idx = np.flatnonzero(patients["subject"].isin(val_ids).to_numpy())
    folds.append((train_idx, val_idx))

print("Using", len(folds), "saved validation folds; held-out test subjects untouched.")

## 2. Training helper

Function to implement at every model to train the model and save the output. 

In [ ]:
# RUN CODE: Define the reusable training helper.
def train_and_save(model_name, make_model):
    for fold_number, (train_idx, val_idx) in enumerate(folds, start=1):
        model = make_model()  # New, unfitted model for this fold.
        model.fit(X.iloc[train_idx], y.iloc[train_idx])  # Train only on training subjects.
        predictions = np.asarray(model.predict(X.iloc[val_idx])).reshape(-1)

        # Save actual values and predictions for the results teammate.
        output = pd.DataFrame({
            "subject": patients.iloc[val_idx]["subject"].to_numpy(),
            "fold": fold_number,
            "actual_UPDRSM": y.iloc[val_idx].to_numpy(),
            "predicted_UPDRSM": predictions
        })
        output.to_csv(
            OUT / "predictions" / f"{model_name}_val_fold{fold_number}.csv",
            index=False
        )
        # Save fitted preprocessing steps and model together.
        joblib.dump(
            model, OUT / "weights" / f"{model_name}_val_fold{fold_number}.joblib"
        )
        print(f"{model_name}: saved validation fold {fold_number} ({len(output)} rows)")

## 3. Mean baseline 
This is used as a reference point. Ideally, our models should perform better than this. We predict the training-fold mean UPDRSM for each validation subject.

In [ ]:
train_and_save("mean_baseline", lambda: DummyRegressor(strategy="mean"))

## 4. Multiple Linear Regression

In [ ]:
train_and_save("linear_regression", make_linear_regression)

## 5. Random Forest

In [ ]:
train_and_save("random_forest", make_random_forest)

## 6. 1D-CNN Regression